# ENGRAM: final Ego4D test of the new search

**The question this run answers:** with the final diary writer (prompt lines-v3, no SEEN line) does the new search
(keyword + meaning + image) find the asked-about moment much more often than the old keyword search?
The v4 run said yes (+28 right moments), but its diary had a SEEN line, which was then dropped.

**What it does:** 20 **new** Ego4D clips (the 63 used before are skipped) → trigger → big model writes the diary →
every question is answered by each search variant on the same diary, plus the big model alone. ~6 hours.

**What we do with the result:** these become the Ego4D numbers of the final system. Later search improvements are
tested on the laptop from the saved diaries, without Kaggle.

**Settings (right panel):** Accelerator **GPU T4** · Internet **On** · Input: dataset **mnemofyp** (latest version)
· **Add-ons → Secrets**: tick `Access Key ID`, `Secret Access Key`, `Region` (keys expire: renew them if the download fails).

**How to run:** **Save Version** → **Save & Run All (Commit)** → Save. Download `search_results.zip` from **Output**.

## 1. Code + packages (~3 min)

In [ ]:
import os, shutil, glob, sys, subprocess
WORK = '/kaggle/working'
CODE = f'{WORK}/mnemo'
code_dir = code_zip = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'mnemo.py' in files and 'modules' in dirs:
        code_dir = root
    if 'mnemo_code.zip' in files:
        code_zip = os.path.join(root, 'mnemo_code.zip')
print('code folder:', code_dir, '| zip:', code_zip)
assert code_dir or code_zip, 'Code not found: add the mnemofyp dataset (Add Input) and wait until it is attached'
shutil.rmtree(CODE, ignore_errors=True)
if code_dir:
    shutil.copytree(code_dir, CODE)
else:
    os.makedirs(CODE)
    !cd "{CODE}" && unzip -q "{code_zip}"
%cd {CODE}
assert os.path.exists('modules/text_embed.py'), 'OLD code: upload the new mnemo_code.zip as a new version of mnemofyp'
!pip install -q -U "transformers>=4.57" accelerate mediapipe ultralytics ego4d "open_clip_torch==3.3.0" "sentence-transformers==5.1.1"
import torch, numpy, transformers
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: switch Accelerator to GPU T4!')
print('transformers', transformers.__version__, '| numpy', numpy.__version__)

## 2. Ego4D access + questions (~1 min)
The keys come from Kaggle Secrets and are written to `~/.aws` (not to the output).

In [ ]:
if True:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    aws = os.path.expanduser('~/.aws')
    os.makedirs(aws, exist_ok=True)
    with open(f'{aws}/credentials', 'w') as f:
        f.write('[default]\n'
                f'aws_access_key_id = {s.get_secret("Access Key ID")}\n'
                f'aws_secret_access_key = {s.get_secret("Secret Access Key")}\n')
    with open(f'{aws}/config', 'w') as f:
        f.write(f'[default]\nregion = {s.get_secret("Region")}\n')
    os.chmod(f'{aws}/credentials', 0o600)

    E4 = '/tmp/ego4d'                 # NOT in /kaggle/working: videos must not end up in the output
    !ego4d --output_directory {E4} --datasets annotations --benchmarks EM -y > /tmp/ego4d_annotations.log 2>&1
    found = glob.glob(f'{E4}/**/nlq_val.json', recursive=True)
    if found:
        NLQ_FILE = found[0]
        print('questions file:', NLQ_FILE)
    else:                              # show why, and still run Part A (it does not need Ego4D)
        print('Ego4D download FAILED. Last lines of the log (send these to Claude):')
        !tail -n 25 /tmp/ego4d_annotations.log | sed -E 's/(AKIA|ASIA)[A-Z0-9]{12,}/<key hidden>/g'
        !which ego4d || echo 'ego4d command not installed'
        raise RuntimeError('Ego4D download failed (see the log above). Expired keys? Get new ones from the Ego4D licence page and update the Kaggle Secrets.')

## 2b. Load the models (~5 min)

In [ ]:
import time, json
import config as C
from modules.vlm_logger import VLMDescriber
from modules.hoi_gate import HOIGate
from modules.landmarks import ClipEmbedder
from modules.text_embed import TextEmbedder
t = time.time()
describer = VLMDescriber(C.VLM_MODEL)
gate = HOIGate('models/hoi_tcn.pt')
clip = ClipEmbedder()          # image fingerprints
text = TextEmbedder()          # meaning search
RESULTS = f'{WORK}/search_results'
print(f'loaded in {time.time() - t:.0f}s | prompt {C.VLM_PROMPT_VERSION} | SEEN line {C.VLM_SEEN} | trigger threshold {C.GATE_THRESHOLD}')
assert C.VLM_PROMPT_VERSION == 'lines-v3' and not C.VLM_SEEN

## 3. Download 20 new Ego4D clips (~72 MB each)
The first 63 clips of this selection were used before (3 to fix the search, 20 for the first test, 20 for the trigger test, 20 for the v4 test), so they are skipped.

In [ ]:
if True:
    N_FULL, N_EXTRA, SKIP, SEED = 20, 0, 63, 0
    !python -m tools.fetch_ego4d --nlq "{NLQ_FILE}" --out {E4} --clips {N_FULL} --extra {N_EXTRA} --seed {SEED} --skip {SKIP}
    SEL = json.load(open(f'{E4}/selected.json'))
    FULL = [c for c in SEL if c['full']]
    EXTRA = [c for c in SEL if not c['full']]
    print(len(FULL), 'full clips,', len(EXTRA), 'trigger-only clips')

## 4. Ego4D questions with every search part (~6 h)
Finished clips are cached.

In [ ]:
if True:
    from evaluation.ego4d_nlq import load_nlq, run_nlq
    from IPython.display import Markdown, display
    EGO_RESULTS = f'{RESULTS}/ego4d'
    nlq = load_nlq(NLQ_FILE)
    ego_results, ego_md = run_nlq(FULL, nlq, describer, gate, EGO_RESULTS, embedder=clip, text=text)
    display(Markdown(ego_md))

## 5. Save
Download `search_results.zip` (tables, every answer, diaries, cached model text and image fingerprints; no video).

In [ ]:
shutil.make_archive(f'{WORK}/search_results', 'zip', RESULTS)
os.chdir(WORK)
from IPython.display import FileLink
display(FileLink('search_results.zip'))